In [191]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [192]:
clinical = pd.read_csv('datasets/clinical/clinical.tsv', sep='\t')
family_his = pd.read_csv('datasets/clinical/family_history.tsv', sep='\t')
exposure = pd.read_csv('datasets/clinical/exposure.tsv', sep='\t')
pathology = pd.read_csv('datasets/clinical/pathology_detail.tsv', sep='\t')


In [193]:
clinical.head()

,project.project_id,cases.case_id,cases.consent_type,cases.days_to_consent,cases.days_to_lost_to_followup,cases.disease_type,cases.index_date,cases.lost_to_followup,cases.primary_site,cases.submitter_id,...,treatments.treatment_duration,treatments.treatment_effect,treatments.treatment_effect_indicator,treatments.treatment_frequency,treatments.treatment_id,treatments.treatment_intent_type,treatments.treatment_or_therapy,treatments.treatment_outcome,treatments.treatment_outcome_duration,treatments.treatment_type
0,TCGA-PAAD,01775b06-5836-469c-8537-120cb8cc94e9,Informed Consent,0,'--,Ductal and Lobular Neoplasms,Diagnosis,No,Pancreas,TCGA-IB-7897,...,'--,'--,'--,'--,11f8ab35-d43b-4a86-adaf-31a177ed4863,Adjuvant,no,'--,'--,"Radiation Therapy, NOS"
1,TCGA-PAAD,01775b06-5836-469c-8537-120cb8cc94e9,Informed Consent,0,'--,Ductal and Lobular Neoplasms,Diagnosis,No,Pancreas,TCGA-IB-7897,...,'--,'--,'--,'--,747e7279-7271-5c39-a57a-5df3bc6145d4,Adjuvant,no,'--,'--,"Pharmaceutical Therapy, NOS"
2,TCGA-PAAD,01775b06-5836-469c-8537-120cb8cc94e9,Informed Consent,0,'--,Ductal and Lobular Neoplasms,Diagnosis,No,Pancreas,TCGA-IB-7897,...,'--,'--,'--,'--,e24e0cfd-21f6-4b4d-8c08-806806910dc9,'--,yes,'--,'--,Whipple
3,TCGA-PAAD,01775b06-5836-469c-8537-120cb8cc94e9,Informed Consent,0,'--,Ductal and Lobular Neoplasms,Diagnosis,No,Pancreas,TCGA-IB-7897,...,'--,'--,'--,'--,27dd51df-86f7-4e6e-8831-9577fd5530cc,'--,unknown,'--,'--,"Radiation Therapy, NOS"
4,TCGA-PAAD,01775b06-5836-469c-8537-120cb8cc94e9,Informed Consent,0,'--,Ductal and Lobular Neoplasms,Diagnosis,No,Pancreas,TCGA-IB-7897,...,'--,'--,'--,'--,a8843746-4194-480b-9c07-7ef9d7216c0b,'--,unknown,'--,'--,"Pharmaceutical Therapy, NOS"


## Reading CSV

In [194]:
# read the json
clinical = pd.read_json('datasets/jsondata/clinical.project-tcga-paad.2025-03-28.json')

In [195]:
clinical =  clinical[clinical["lost_to_followup"] == "No"]

In [196]:
clinical["disease_type"].value_counts()

disease_type
Ductal and Lobular Neoplasms             118
Adenomas and Adenocarcinomas              24
Cystic, Mucinous and Serous Neoplasms      4
Epithelial Neoplasms, NOS                  1
Name: count, dtype: int64

In [197]:
clinical = clinical.drop(
    columns=["lost_to_followup", "updated_datetime", "case_id", "project", "diagnoses", "consent_type", "submitter_id" , "primary_site" , "index_date" , "state"])

In [198]:
clinical.head()

,family_histories,disease_type,days_to_consent,demographic,exposures,follow_ups
0,"[{'relative_with_cancer_history': 'yes', 'upda...",Ductal and Lobular Neoplasms,0.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'alcohol_history': 'Yes', 'updated_datetime'...","[{'timepoint_category': 'Follow-up', 'follow_u..."
1,"[{'relative_with_cancer_history': 'unknown', '...",Ductal and Lobular Neoplasms,28.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_status': 'Unknown', 'update...","[{'timepoint_category': 'Follow-up', 'follow_u..."
2,"[{'relative_with_cancer_history': 'unknown', '...",Ductal and Lobular Neoplasms,5.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_status': 'Unknown', 'update...","[{'timepoint_category': 'Last Contact', 'follo..."
3,"[{'relative_with_cancer_history': 'unknown', '...",Ductal and Lobular Neoplasms,48.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_status': 'Unknown', 'update...","[{'timepoint_category': 'Follow-up', 'follow_u..."
4,"[{'relative_with_cancer_history': 'yes', 'upda...",Ductal and Lobular Neoplasms,-31.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_quit_year': 1979, 'tobacco_...","[{'timepoint_category': 'Last Contact', 'follo..."


In [199]:
def array_to_obj_features(df, col):
    """
    Convert a column of array-like strings to a column of objects.
    """
    df[col] = df[col].apply(lambda x: eval(x) if isinstance(x, str) else x)
    return df

In [200]:
df = clinical.copy()

In [201]:
import pandas as pd


def process_family_history(row):
    ans = ""

    # Ensure row is a valid list
    if isinstance(row, list):
        for i in range(len(row)):
            # Ensure row[i] is a dictionary before accessing keys
            if isinstance(row[i], dict):
                if row[i].get("relative_with_cancer_history") not in ["unknown", "no"]:
                    ans += str(row[i].get("relationship_primary_diagnosis", "")) + ","

    return ans[:-1] if ans else ""  # Remove trailing comma if any

In [202]:
df["family_histories"] = df["family_histories"].apply(process_family_history)

In [203]:
# # if the family history is empty, set it to Cancer
df["family_histories"] = df["family_histories"].replace("", "Cancer")

In [204]:
df.head()

,family_histories,disease_type,days_to_consent,demographic,exposures,follow_ups
0,"Melanoma,Cancer",Ductal and Lobular Neoplasms,0.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'alcohol_history': 'Yes', 'updated_datetime'...","[{'timepoint_category': 'Follow-up', 'follow_u..."
1,Cancer,Ductal and Lobular Neoplasms,28.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_status': 'Unknown', 'update...","[{'timepoint_category': 'Follow-up', 'follow_u..."
2,Cancer,Ductal and Lobular Neoplasms,5.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_status': 'Unknown', 'update...","[{'timepoint_category': 'Last Contact', 'follo..."
3,Cancer,Ductal and Lobular Neoplasms,48.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_status': 'Unknown', 'update...","[{'timepoint_category': 'Follow-up', 'follow_u..."
4,Cancer,Ductal and Lobular Neoplasms,-31.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_quit_year': 1979, 'tobacco_...","[{'timepoint_category': 'Last Contact', 'follo..."


### Demographic

In [205]:
def extract_demographic(row, feature):
    return row[feature] if isinstance(row, dict) and feature in row else None

In [206]:
features_to_take = ["cause_of_death", "ethnicity", "gender", "race", "vital_status", "days_to_death"]

for feature in features_to_take:
    df[feature] = df["demographic"].apply(lambda x: extract_demographic(x, feature))

In [207]:

df = df.drop(columns=["days_to_death"])

In [208]:
df["cause_of_death"] = df["cause_of_death"].replace("None", "Cancer Related")

In [209]:
cod_df =  df[df["cause_of_death"].isna()]

In [210]:
cod_df["cause_of_death"] = "Cancer Related"

C:\Users\prans\AppData\Local\Temp\ipykernel_24116\509288296.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cod_df["cause_of_death"] = "Cancer Related"


In [211]:
# merge the two dataframes
df = pd.concat([df, cod_df], ignore_index=True)

In [212]:
df = df.dropna()

In [213]:
df.head()

,family_histories,disease_type,days_to_consent,demographic,exposures,follow_ups,cause_of_death,ethnicity,gender,race,vital_status
0,"Melanoma,Cancer",Ductal and Lobular Neoplasms,0.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'alcohol_history': 'Yes', 'updated_datetime'...","[{'timepoint_category': 'Follow-up', 'follow_u...",Cancer Related,not hispanic or latino,female,white,Dead
1,Cancer,Ductal and Lobular Neoplasms,28.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_status': 'Unknown', 'update...","[{'timepoint_category': 'Follow-up', 'follow_u...",Cancer Related,not hispanic or latino,female,white,Dead
2,Cancer,Ductal and Lobular Neoplasms,5.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_status': 'Unknown', 'update...","[{'timepoint_category': 'Last Contact', 'follo...",Cancer Related,not hispanic or latino,female,white,Dead
3,Cancer,Ductal and Lobular Neoplasms,48.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_status': 'Unknown', 'update...","[{'timepoint_category': 'Follow-up', 'follow_u...",Cancer Related,not hispanic or latino,female,black or african american,Dead
4,Cancer,Ductal and Lobular Neoplasms,-31.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_quit_year': 1979, 'tobacco_...","[{'timepoint_category': 'Last Contact', 'follo...",Cancer Related,not hispanic or latino,male,white,Dead


### Exposures

In [214]:
# def process_exposures(row,feature):
#     result = {}

#     # Ensure row is a valid list
#     if isinstance(row, list):
#         for item in row:
#             print(item)
#             # Ensure item is a dictionary before accessing keys
#             if isinstance(item, dict):
#                 if "exposure_type" in item:
#                     return item.get("exposure_type", "")
#                 if "alcohol_intensity" in item:
#                     return item.get("alcohol_intensity", "")
#                 if "tobacco_smoking_status" in item:
#                     return item.get("tobacco_smoking_status", "")

#     return ""

In [215]:
# features_exposure_arr = ["exposure_type", "alcohol_intensity", "tobacco_smoking_status"]

# for feature in features_exposure_arr:
#     df[feature] = df["exposures"].apply(lambda x: process_exposures(x , feature))

In [216]:
df.head()

,family_histories,disease_type,days_to_consent,demographic,exposures,follow_ups,cause_of_death,ethnicity,gender,race,vital_status
0,"Melanoma,Cancer",Ductal and Lobular Neoplasms,0.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'alcohol_history': 'Yes', 'updated_datetime'...","[{'timepoint_category': 'Follow-up', 'follow_u...",Cancer Related,not hispanic or latino,female,white,Dead
1,Cancer,Ductal and Lobular Neoplasms,28.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_status': 'Unknown', 'update...","[{'timepoint_category': 'Follow-up', 'follow_u...",Cancer Related,not hispanic or latino,female,white,Dead
2,Cancer,Ductal and Lobular Neoplasms,5.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_status': 'Unknown', 'update...","[{'timepoint_category': 'Last Contact', 'follo...",Cancer Related,not hispanic or latino,female,white,Dead
3,Cancer,Ductal and Lobular Neoplasms,48.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_status': 'Unknown', 'update...","[{'timepoint_category': 'Follow-up', 'follow_u...",Cancer Related,not hispanic or latino,female,black or african american,Dead
4,Cancer,Ductal and Lobular Neoplasms,-31.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_quit_year': 1979, 'tobacco_...","[{'timepoint_category': 'Last Contact', 'follo...",Cancer Related,not hispanic or latino,male,white,Dead


In [217]:
df["disease_reponse"] = df["follow_ups"].apply(lambda x: x[0]["disease_response"] if isinstance(x, list) and len(x) > 0 and "disease_response" in x[0] else None)

In [218]:
df[df["disease_reponse"].isna()]["disease_reponse"] = "Unknown"

C:\Users\prans\AppData\Local\Temp\ipykernel_24116\2031510656.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[df["disease_reponse"].isna()]["disease_reponse"] = "Unknown"


In [221]:
dr_with_none = df[df["disease_reponse"].isna()]

dr_with_none["disease_reponse"] = "Unknown"

C:\Users\prans\AppData\Local\Temp\ipykernel_24116\1425697923.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dr_with_none["disease_reponse"] = "Unknown"


In [222]:
# merge the two dataframes

df = pd.concat([df, dr_with_none], ignore_index=True)

In [ ]:
df = df.dropna()

In [227]:
df.head()

,family_histories,disease_type,days_to_consent,demographic,exposures,follow_ups,cause_of_death,ethnicity,gender,race,vital_status,disease_reponse
0,"Melanoma,Cancer",Ductal and Lobular Neoplasms,0.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'alcohol_history': 'Yes', 'updated_datetime'...","[{'timepoint_category': 'Follow-up', 'follow_u...",Cancer Related,not hispanic or latino,female,white,Dead,WT-With Tumor
1,Cancer,Ductal and Lobular Neoplasms,28.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_status': 'Unknown', 'update...","[{'timepoint_category': 'Follow-up', 'follow_u...",Cancer Related,not hispanic or latino,female,white,Dead,WT-With Tumor
2,Cancer,Ductal and Lobular Neoplasms,5.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_status': 'Unknown', 'update...","[{'timepoint_category': 'Last Contact', 'follo...",Cancer Related,not hispanic or latino,female,white,Dead,WT-With Tumor
3,Cancer,Ductal and Lobular Neoplasms,48.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_status': 'Unknown', 'update...","[{'timepoint_category': 'Follow-up', 'follow_u...",Cancer Related,not hispanic or latino,female,black or african american,Dead,WT-With Tumor
4,Cancer,Ductal and Lobular Neoplasms,-31.0,"{'cause_of_death': 'Cancer Related', 'ethnicit...","[{'tobacco_smoking_quit_year': 1979, 'tobacco_...","[{'timepoint_category': 'Last Contact', 'follo...",Cancer Related,not hispanic or latino,male,white,Dead,WT-With Tumor


In [228]:
df = df.drop(columns=["demographic", "exposures", "follow_ups"])

In [229]:
df.head()

,family_histories,disease_type,days_to_consent,cause_of_death,ethnicity,gender,race,vital_status,disease_reponse
0,"Melanoma,Cancer",Ductal and Lobular Neoplasms,0.0,Cancer Related,not hispanic or latino,female,white,Dead,WT-With Tumor
1,Cancer,Ductal and Lobular Neoplasms,28.0,Cancer Related,not hispanic or latino,female,white,Dead,WT-With Tumor
2,Cancer,Ductal and Lobular Neoplasms,5.0,Cancer Related,not hispanic or latino,female,white,Dead,WT-With Tumor
3,Cancer,Ductal and Lobular Neoplasms,48.0,Cancer Related,not hispanic or latino,female,black or african american,Dead,WT-With Tumor
4,Cancer,Ductal and Lobular Neoplasms,-31.0,Cancer Related,not hispanic or latino,male,white,Dead,WT-With Tumor


In [230]:
df.to_csv("datasets/clinical/clinical_cleaned.csv", index=False)